# Tokenization Comparison

- Creator: Sebastian Birk (sebastian.birk@helmholtz-munich.de).
- Date of Creation: 05.06.2024
- Date of Last Modification: 09.07.2024

## 1. Setup

### 1.1 Import Libraries

In [ ]:
import os
import warnings

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

### 1.2 Run Notebook Setup

In [ ]:
warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)

### 1.3 Configure Paths and Create Directories

In [ ]:
# Define paths
st_data_folder_path = "../../datasets/st_data" # spatially resolved transcriptomics data
st_data_bronze_folder_path = f"{st_data_folder_path}/bronze"
st_data_silver_folder_path = f"{st_data_folder_path}/silver"
st_data_gold_folder_path = f"{st_data_folder_path}/gold"
results_folder_path = "../../datasets/results"

## 2. Tokenization Comparison

In [ ]:
# Define dataset
#dataset_name = "nanostring_cosmx_human_brain"
#dataset_name = "sim_ref_1354fts_10000locs_8r_8c_20p_3fc"
n_sim = 1
dataset_name = f"sim{n_sim}_1105fts_10000locs"
#dataset_name = "sim_free_1354fts_10000locs_8r_8c_20p_3fc"

In [ ]:
def capitalize_after_space(s):
    return " ".join(word.capitalize() for word in s.split("_"))

### 2.1 Baseline

In [ ]:
# Load data
top_n = 10 # 1090, 30, 10
mode = "randpos" # "randpos", "sinpos", "nopos"
randomize = False

tokenizer = "cell_graph_tokenizer"
df_gt = pd.read_csv(f"{results_folder_path}/{tokenizer}/{dataset_name}/clf_scores_baseline_top{top_n}_{mode}{'_randomized' if randomize else ''}.csv")
tokenizer = "cell_neighborhood_tokenizer"
df_nt = pd.read_csv(f"{results_folder_path}/{tokenizer}/{dataset_name}/clf_scores_baseline_top{top_n}_{mode}{'_randomized' if randomize else ''}.csv")
df = pd.concat([df_gt, df_nt], ignore_index=True)

# Only keep test scores
df = df[df["clf_category"] == "test"]

# Rename token types
df["token_type"] = df["token_type"].replace(
    {"cell": "Index Cell Tokens", "neighbor_cells": "Neighbor Cell Tokens"})

# Rename classification tasks
df["clf_task"] = df["clf_task"].replace(
    {"cell_types": "Cell Type Classification", "niche_types": "Niche Type Classification"})

# Create a combined hue for clf_task and clf_category
df["token_task"] = df["token_type"] + " - " + df["clf_task"]

# Create subplots
fig, axes = plt.subplots(1, 2, figsize=(14, 7), sharey=True)

# Plot for Cell Graph Tokenizer
sns.barplot(
    y="f1_score", x="normalization", hue="token_task", data=df[df["tokenizer"] == "cell_neighborhood_tokenizer"], ci="sd", 
    palette="Set3", orient="v", width=0.6, ax=axes[0]
)
axes[0].set_title("Cell Neighborhood Tokenizer", fontsize=16, fontweight="bold")
axes[0].set_ylabel("F1 Score")
axes[0].set_xlabel("Normalization Method")
axes[0].legend_.remove()  # Remove legend from the first subplot
axes[0].grid(True, which='both', linestyle='--', linewidth=0.5, alpha=0.7)
axes[0].tick_params(axis='x', rotation=90)  # Rotate x-axis labels

# Plot for Cell Neighborhood Tokenizer
sns.barplot(
    y="f1_score", x="normalization", hue="token_task", data=df[df["tokenizer"] == "cell_graph_tokenizer"], ci="sd", 
    palette="Set3", orient="v", width=0.6, ax=axes[1]
)
axes[1].set_title("Cell Graph Tokenizer", fontsize=16, fontweight="bold")
axes[1].set_ylabel("F1 Score")
axes[1].set_xlabel("Normalization Method")
axes[1].grid(True, which='both', linestyle='--', linewidth=0.5, alpha=0.7)
axes[1].tick_params(axis='x', rotation=90)  # Rotate x-axis labels
axes[1].legend_.remove()  # Remove legend from the second subplot

# Add a single legend below both plots
handles, labels = axes[1].get_legend_handles_labels()
fig.legend(handles, labels, title="Token Type - Classification Task", bbox_to_anchor=(0.5, -0.05), loc="upper center", ncol=2)

# Overall title for the figure
fig.suptitle(
    f"""{'Randomized' if randomize else f'Top {top_n} Token'} Random Gene Embeddings{
    ' with Random Positional Embeddings' if mode == 'randpos' else ' with Sinusoidal Positional Embeddings' if mode == 'sinpos' else ''}
    F1 Score - {'Nanostring CosMx Human Brain' if dataset_name == 'nanostring_cosmx_human_brain'
    else f'Simulation {n_sim}'}""", fontsize=20, fontweight="bold", y=0.9)

# Adjust layout
fig.tight_layout(rect=[0, -0.05, 1., 0.9])  # Adjust layout to make room for the legend

os.makedirs(f"{results_folder_path}/{dataset_name}/", exist_ok=True)
plt.savefig(f"{results_folder_path}/{dataset_name}/clf_scores_baseline_top{top_n}_{mode}{'_randomized' if randomize else ''}.svg", bbox_inches="tight")
plt.show()

#### 2.2 scGPT

In [ ]:
if "sim" not in dataset_name:
    # scGPT Embeddings don't make sense with simulated data because of random gene assignments
    
    # Load data
    top_n = 1304
    
    tokenizer = "cell_graph_tokenizer"
    df_gt = pd.read_csv(f"{results_folder_path}/{tokenizer}/{dataset_name}/clf_scores_scgpt_top{top_n}.csv")
    tokenizer = "cell_neighborhood_tokenizer"
    df_nt = pd.read_csv(f"{results_folder_path}/{tokenizer}/{dataset_name}/clf_scores_scgpt_top{top_n}.csv")
    df = pd.concat([df_gt, df_nt], ignore_index=True)
    
    # Only keep test scores
    df = df[df["clf_category"] == "test"]
    
    # Rename token types
    df["token_type"] = df["token_type"].replace(
        {"cell": "Index Cell Tokens", "neighbor_cells": "Neighbor Cell Tokens"})
    
    # Rename classification tasks
    df["clf_task"] = df["clf_task"].replace(
        {"cell_types": "Cell Type Classification", "niche_types": "Niche Type Classification"})
    
    # Create a combined hue for clf_task and clf_category
    df["token_task"] = df["token_type"] + " - " + df["clf_task"]
    
    # Create subplots
    fig, axes = plt.subplots(1, 2, figsize=(14, 7), sharey=True)
    
    # Plot for Cell Graph Tokenizer
    sns.barplot(
        y="f1_score", x="normalization", hue="token_task", data=df[df["tokenizer"] == "cell_neighborhood_tokenizer"], ci="sd", 
        palette="Set3", orient="v", width=0.6, ax=axes[0]
    )
    axes[0].set_title("Cell Neighborhood Tokenizer", fontsize=16, fontweight="bold")
    axes[0].set_ylabel("F1 Score")
    axes[0].set_xlabel("Normalization Method")
    axes[0].legend_.remove()  # Remove legend from the first subplot
    axes[0].grid(True, which='both', linestyle='--', linewidth=0.5, alpha=0.7)
    axes[0].tick_params(axis='x', rotation=90)  # Rotate x-axis labels
    
    # Plot for Cell Neighborhood Tokenizer
    sns.barplot(
        y="f1_score", x="normalization", hue="token_task", data=df[df["tokenizer"] == "cell_graph_tokenizer"], ci="sd", 
        palette="Set3", orient="v", width=0.6, ax=axes[1]
    )
    axes[1].set_title("Cell Graph Tokenizer", fontsize=16, fontweight="bold")
    axes[1].set_ylabel("F1 Score")
    axes[1].set_xlabel("Normalization Method")
    axes[1].grid(True, which='both', linestyle='--', linewidth=0.5, alpha=0.7)
    axes[1].tick_params(axis='x', rotation=90)  # Rotate x-axis labels
    axes[1].legend_.remove()  # Remove legend from the second subplot
    
    # Add a single legend below both plots
    handles, labels = axes[1].get_legend_handles_labels()
    fig.legend(handles, labels, title="Token Type - Classification Task", bbox_to_anchor=(0.5, -0.05), loc="upper center", ncol=2)
    
    # Overall title for the figure
    fig.suptitle(
        f"Top 10 Token scGPT Embeddings F1 Score - {'Nanostring CosMx Human Brain' if dataset_name == 'nanostring_cosmx_human_brain' else 'Reference-based Simulation' if 'ref' in dataset_name else 'Free Simulation'}", fontsize=20, fontweight="bold", y=0.9)
    
    # Adjust layout
    fig.tight_layout(rect=[0, -0.05, 1., 0.9])  # Adjust layout to make room for the legend
    
    plt.savefig(f"{results_folder_path}/{dataset_name}/clf_scores_scgpt.svg", bbox_inches="tight")
    plt.show()